# Day 30: LLM Evaluation and QA
**Golden datasets, prompt regression testing, context precision and recall, BLEU, and quality gates**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.

## 1. Setup

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai scikit-learn numpy pandas pytest

# ===== Option B: VS Code / local =====
# pip install openai scikit-learn numpy pandas pytest
# Create a .env file or set the variable in your terminal:  OPENAI_API_KEY=sk-...
# No key? Everything still runs in OFFLINE demo mode with a tiny built-in stand-in model.
print("Setup done")

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
"""
import json, os, re
import numpy as np

def _get_key():
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")                 # change to any current chat model you have access to
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None):
    if ONLINE:
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}])
        return r.choices[0].message.content
    return offline_llm(prompt, system)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, ONLINE

## 2. The system under test, with versioned configs

In [ ]:
%%writefile knowledge.md
# Sakura Logistics Employee Handbook (sample)

## 1. Working hours
Standard working hours are 9:30 to 18:30, Monday to Friday. Teams in the Tokyo office follow JST and teams in the Bengaluru office follow IST.
Flexible start times between 8:00 and 11:00 are allowed with manager approval.

## 2. Leave policy
Full-time employees receive 18 days of paid leave per year. Leave accrues monthly at 1.5 days.
Unused leave can be carried forward up to a maximum of 10 days. Carried forward leave expires on 31 March of the following year.
Sick leave is separate: employees receive 8 days of paid sick leave per year. A medical certificate is required for sick leave longer than 2 consecutive days.

## 3. Remote work
Employees may work remotely up to 2 days per week. Remote work must be logged in the HR portal by 10:00 on the same day.
Remote work from another country requires approval from HR and the security team at least 14 days in advance.

## 4. Expenses
Travel expenses must be submitted within 30 days with receipts. Meals during business travel are reimbursed up to 2,000 INR or 4,000 JPY per day.
Expenses above 50,000 INR require approval from a director before booking.

## 5. Security
Laptops must use full disk encryption. Customer data must never be copied to personal devices or uploaded to unapproved AI tools.
Security incidents must be reported to security@sakura-logistics.example within 1 hour of discovery.

## IT FAQ
Error VPN-417 means your VPN certificate has expired. Renew it from the IT self-service portal and restart the VPN client.
Password resets are done through the self-service portal. Passwords must be at least 14 characters long.
Approved AI tools are listed on the IT intranet page. Do not paste customer data into tools that are not on that list.
New laptops are issued within 5 working days of a request approved by your manager.

## Travel FAQ
Book flights through the corporate travel desk at least 10 days before travel. Economy class is standard for flights under 6 hours.
Hotel stays are reimbursed up to 6,000 INR per night in India and 15,000 JPY per night in Japan.
Visa support letters are issued by HR within 3 working days.

In [ ]:
%%writefile rag_system.py
import re, numpy as np
from genai_helpers import llm, embed

TEXT = open("knowledge.md").read()
CHUNKS = [s.strip() for s in re.split(r"(?<=[.!?])\s+", re.sub(r"#+ .*\n", "", TEXT)) if s.strip()]
IDS = [f"c{i + 1}" for i in range(len(CHUNKS))]
VECS = embed(CHUNKS)

CONFIGS = {
    "v1": {"k": 3, "system": "Answer using ONLY the context. If the answer is not there, say \"I don't know\"."},
    "v2": {"k": 1, "system": "Answer briefly using ONLY the context. If the answer is not there, say \"I don't know\"."},  # "save cost"
}

def retrieve(question, k):
    order = np.argsort(VECS @ embed(question)[0])[::-1][:k]
    return [IDS[i] for i in order], [CHUNKS[i] for i in order]

def answer(question, version="v1"):
    cfg = CONFIGS[version]
    ids, ctx = retrieve(question, cfg["k"])
    out = llm("Context:\n" + "\n".join(ctx) + f"\n\nQuestion: {question}", system=cfg["system"], temperature=0)
    return out, ids

## 3. A golden dataset (versioned JSONL)

In [ ]:
import json, importlib, rag_system
importlib.reload(rag_system)
from rag_system import CHUNKS, IDS, answer, retrieve

def cid(fragment):
    return [IDS[i] for i, c in enumerate(CHUNKS) if fragment in c]

GOLDEN = [
    ("leave-001", "How many paid leave days do full-time staff get?", "18 days", ["18 days"], ["8 days"], "18 days of paid leave", "leave"),
    ("leave-002", "How many sick days do I get each year?", "8 days", ["8 days"], ["18 days"], "8 days of paid sick leave", "leave"),
    ("leave-003", "When does carried over leave expire?", "31 March", ["31 March"], ["30 June"], "expires on 31 March", "leave"),
    ("remote-001", "How many days a week can I work remotely?", "2 days", ["2 days"], ["5 days"], "up to 2 days per week", "remote"),
    ("remote-002", "How far ahead must I request remote work abroad?", "14 days", ["14 days"], [], "at least 14 days in advance", "remote"),
    ("travel-001", "Hotel limit per night in Japan?", "15,000 JPY", ["15,000 JPY"], ["6,000 INR per night in Japan"], "15,000 JPY", "travel"),
    ("travel-002", "Deadline for submitting travel expenses?", "30 days", ["30 days"], [], "within 30 days", "travel"),
    ("travel-003", "How early should flights be booked?", "10 days", ["10 days"], [], "10 days before travel", "travel"),
    ("it-001", "What does error VPN-417 mean?", "VPN certificate has expired", ["certificate"], [], "VPN-417", "it"),
    ("it-002", "Minimum password length?", "14 characters", ["14 characters"], [], "14 characters", "it"),
    ("it-003", "How long does a new laptop take?", "5 working days", ["5 working days"], [], "5 working days", "it"),
    ("sec-001", "How fast must I report a security incident?", "1 hour", ["1 hour"], ["24 hours"], "within 1 hour", "security"),
]
with open("golden_v1.jsonl", "w") as f:
    for gid, q, exp, inc, exc, frag, topic in GOLDEN:
        f.write(json.dumps({"id": gid, "question": q, "expected": exp, "must_include": inc, "must_not_include": exc,
                            "relevant_ids": cid(frag), "tags": {"topic": topic, "lang": "en"}, "version": 1}) + "\n")
golden = [json.loads(l) for l in open("golden_v1.jsonl")]
print(len(golden), "golden items. Example:\n", json.dumps(golden[2], indent=2))

## 4. Prompt regression tests with PyTest
The version under test comes from an environment variable, exactly as a CI pipeline would set it.

In [ ]:
%%writefile test_golden.py
import json, os, pytest
from rag_system import answer

VERSION = os.getenv("PROMPT_VERSION", "v1")
GOLDEN = [json.loads(l) for l in open("golden_v1.jsonl")]

@pytest.mark.parametrize("item", GOLDEN, ids=lambda i: i["id"])
def test_golden_item(item):
    out, _ = answer(item["question"], VERSION)
    for phrase in item["must_include"]:
        assert phrase.lower() in out.lower(), f"missing '{phrase}' in: {out}"
    for phrase in item["must_not_include"]:
        assert phrase.lower() not in out.lower(), f"forbidden '{phrase}' in: {out}"

In [ ]:
# ===== Option A: Google Colab (active) =====
!PROMPT_VERSION=v1 python -m pytest -q test_golden.py 2>&1 | tail -3
!PROMPT_VERSION=v2 python -m pytest -q test_golden.py 2>&1 | tail -6

# ===== Option B: VS Code terminal =====
# Windows PowerShell:  $env:PROMPT_VERSION="v2"; pytest -q test_golden.py
# macOS/Linux:          PROMPT_VERSION=v2 pytest -q test_golden.py

### Per-item comparison: what exactly regressed?

In [ ]:
import pandas as pd
def passes(item, version):
    out, ids = answer(item["question"], version)
    ok = all(p.lower() in out.lower() for p in item["must_include"]) and not any(p.lower() in out.lower() for p in item["must_not_include"])
    return ok, ids

rows = []
for g in golden:
    ok1, _ = passes(g, "v1"); ok2, _ = passes(g, "v2")
    rows.append({"id": g["id"], "v1": ok1, "v2": ok2, "status": "REGRESSED" if ok1 and not ok2 else ("fixed" if ok2 and not ok1 else "")})
cmp = pd.DataFrame(rows)
print(cmp.to_string(index=False))
print(f"\nv1 pass rate {cmp.v1.mean():.2f} | v2 pass rate {cmp.v2.mean():.2f} | regressed: {list(cmp[cmp.status == 'REGRESSED'].id)}")
if not ONLINE:
    print("Offline, the stand-in answers by picking one sentence, so absolute pass rates are low. The comparison between versions is still meaningful.")

## 5. Context precision, recall and average precision

In [ ]:
def precision(ret, rel):
    return sum(c in rel for c in ret) / len(ret)

def recall(ret, rel):
    return len(set(ret) & set(rel)) / len(rel)

def avg_precision(ret, rel):
    hits, total = 0, 0.0
    for k, c in enumerate(ret, 1):
        if c in rel:
            hits += 1
            total += hits / k
    return total / len(rel)

table = []
for k in [1, 3, 5]:
    p, r, ap = [], [], []
    for g in golden:
        ret, _ = retrieve(g["question"], k)
        rel = set(g["relevant_ids"])
        p.append(precision(ret, rel)); r.append(recall(ret, rel)); ap.append(avg_precision(ret, rel))
    table.append({"k": k, "precision": round(sum(p) / len(p), 2), "recall": round(sum(r) / len(r), 2), "avg_precision": round(sum(ap) / len(ap), 2)})
print(pd.DataFrame(table).to_string(index=False))

## 6. BLEU from scratch, and where it misleads

In [ ]:
import math, re
from collections import Counter

def ngrams(tokens, n):
    return Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))

def bleu(candidate, reference, max_n=4):
    c, r = re.findall(r"\w+", candidate.lower()), re.findall(r"\w+", reference.lower())
    logs = []
    for n in range(1, max_n + 1):
        cn, rn = ngrams(c, n), ngrams(r, n)
        overlap = sum(min(cnt, rn[g]) for g, cnt in cn.items())
        logs.append(math.log((overlap + 1) / (max(1, sum(cn.values())) + 1)))      # add-1 smoothing
    bp = 1.0 if len(c) > len(r) else math.exp(1 - len(r) / max(1, len(c)))
    return round(bp * math.exp(sum(logs) / max_n), 2)

ref = "Carried forward leave expires on 31 March of the following year."
cases = {
    "exact copy": ref,
    "correct paraphrase": "Any leave you roll over must be used by the end of next March.",
    "wrong, similar words": "Carried forward leave expires on 30 June of the following year.",
}
for name, cand in cases.items():
    print(f"{name:<22} BLEU {bleu(cand, ref)}")
print("\nThe wrong answer scores far higher than the correct paraphrase. Use BLEU for translation-like tasks, not open answers.")

## 7. A quality gate for CI

In [ ]:
%%writefile quality_gate.py
import json, os, sys
from rag_system import answer, retrieve

VERSION = sys.argv[1] if len(sys.argv) > 1 else "v1"
THRESHOLDS = {"recall": 0.85}          # absolute floor for retrieval
MAX_DROP = 0.05                         # allowed pass-rate drop versus the baseline (v1)
golden = [json.loads(l) for l in open("golden_v1.jsonl")]

passed, recalls = 0, []
for g in golden:
    out, ids = answer(g["question"], VERSION)
    ok = all(p.lower() in out.lower() for p in g["must_include"]) and not any(p.lower() in out.lower() for p in g["must_not_include"])
    passed += ok
    recalls.append(len(set(ids) & set(g["relevant_ids"])) / len(g["relevant_ids"]))

scores = {"pass_rate": round(passed / len(golden), 2), "recall": round(sum(recalls) / len(recalls), 2)}
failures = [f"{k} {scores[k]} < {v}" for k, v in THRESHOLDS.items() if scores[k] < v]
if VERSION != "v1" and os.path.exists("report_v1.json"):
    base = json.load(open("report_v1.json"))["scores"]["pass_rate"]
    if scores["pass_rate"] < base - MAX_DROP:
        failures.append(f"pass_rate dropped {base - scores['pass_rate']:.2f} vs baseline {base}")
print(f"{VERSION}: {'FAIL' if failures else 'PASS'} {scores}" + (f" -> {failures}" if failures else ""))
json.dump({"version": VERSION, "scores": scores, "failures": failures}, open(f"report_{VERSION}.json", "w"), indent=2)
sys.exit(1 if failures else 0)

In [ ]:
# ===== Option A: Google Colab (active) =====
!python quality_gate.py v1 && echo "Gate passed: release allowed"
!python quality_gate.py v2 || echo "Gate failed: release blocked"

# ===== Option B: VS Code terminal / Azure Pipelines step =====
# python quality_gate.py v2      (a non-zero exit code fails the pipeline stage)

### TODO 1: With a real model (API key), add an absolute threshold of `pass_rate >= 0.9` to `THRESHOLDS` and rerun the gate for both versions.

In [ ]:
# TODO: write your code here

---
**Yoi shuumatsu wo** (良い週末を): *Have a great weekend*

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)